# Lab 9.9 &mdash; Privacy: Minimize & Redact

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Redact long identifiers (account/card numbers) from text
- Send the model only the fields the task needs
- Treat data handling as a first-class design constraint

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; Privacy, compliance & data handling](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-09"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
High-stakes agents run on the most sensitive data there is (deck slide 16). Two disciplines: **minimize**
&mdash; send the model only the fields the task needs, not the whole record &mdash; and **redact** &mdash;
mask identifiers (account numbers, card numbers) before they leave your systems. Less exposed data is less
risk. It's why Day 1 started on **local models**; here you also see why Day-5 hosted calls (Groq) must be
fed **minimized, redacted** input.

In [ ]:
# We mask any run of 6+ consecutive digits (account/card numbers), keeping short numbers like a year.
print("redact 'acct 1234567890 for FY2026' -> mask the long number, keep 2026")

## Build it
Complete `redact_account` (mask long digit runs) and `minimize` (keep only needed fields), then run the
cell.

In [ ]:
def redact_account(text):
    out, run = [], ""
    for ch in text + " ":            # trailing space flushes the final run
        if ch.isdigit():
            run += ch
        else:
            if ___:            # TODO: run is a long number (6+ digits)
                out.append("[REDACTED]")
            else:
                out.append(run)
            run = ""
            out.append(ch)
    return "".join(out).rstrip()

def minimize(record, needed_fields):
    # send the model ONLY the fields the task needs -- not the whole record
    return ___   # TODO: build a dict of just the needed_fields that exist in record

try:
    print(redact_account("acct 1234567890 for FY2026"))
    rec = {"name": "ACME", "revenue": 120.0, "account": "1234567890", "ssn": "999-99-9999"}
    print(minimize(rec, ["name", "revenue"]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A 10-digit account number becomes **`[REDACTED]`**; a 4-digit year survives &mdash; the threshold is the rule.
- `minimize` sends **only** `["name", "revenue"]`; the account number and SSN never leave your systems.
- Before any text goes to a hosted model, run it through both &mdash; the model should never see what it doesn't need.

## Your turn (open task &mdash; no grader)
Extend `redact_account` (or write a second redactor) to also mask an email address, then minimize a record
down to the two fields a summary needs. **What good looks like:** the text you'd send a hosted model carries
no raw identifiers, and only the necessary fields leave your boundary.

---
### Key takeaway
Minimize what you send and redact identifiers before they leave your systems. Data handling is a first-class design constraint -- decide where the data can go before you build, which is why this course runs on local models where it can.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>